In [1]:
import cv2
import json
import time
import numpy as np
import pandas as pd
import mediapipe as mp
import tensorflow as tf
from pathlib import Path
from collections import deque

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print(f'GPU detected: {gpus[0].name}')
else:
    print('No GPU — running on CPU')


GPU detected: /physical_device:GPU:0


In [2]:
# Update these paths if your model or classes are named differently!
DIR = Path(r"M:/Term 10/Grad/SLR Main/Words/ArSL Word (Arabic)")
MODEL_PATH = DIR / "arsl_word_lstm_model_final_v1.h5" 
CLASSES_CSV = DIR / "arsl_word_classes.csv"
SCALER_PATH = DIR / "arsl_scaler_stats.npz"

# Live Test configurations
SEQUENCE_LENGTH = 30
CONFIDENCE_THRESHOLD = 0.35
PREDICTION_INTERVAL = 0.5
STABILITY_WINDOW = 3
COOLDOWN_TIME = 2.0
CAMERA_INDEX = 0
CAMERA_WIDTH = 1280
CAMERA_HEIGHT = 720


In [3]:
class TemporalAttention(tf.keras.layers.Layer):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)

    def build(self, input_shape):
        self.W = self.add_weight(name='att_weight', shape=(input_shape[-1], 1),
                                 initializer='glorot_uniform', trainable=True)
        self.b = self.add_weight(name='att_bias', shape=(input_shape[1], 1),
                                 initializer='zeros', trainable=True)

    def call(self, x):
        e = tf.nn.tanh(tf.matmul(x, self.W) + self.b)
        a = tf.nn.softmax(e, axis=1)
        output = tf.reduce_sum(x * a, axis=1)
        return output

print('Loading model...')
model = tf.keras.models.load_model(
    str(MODEL_PATH),
    custom_objects={'TemporalAttention': TemporalAttention}
)
print(f'Model loaded: {model.name}')

NUM_FEATURES = model.input_shape[-1]
NUM_HANDS = 2 if (NUM_FEATURES == 126 or NUM_FEATURES == 258) else 1
LANDMARKS_PER_HAND = 63

if SCALER_PATH.exists():
    z = np.load(SCALER_PATH)
    scaler_mean = z['mean']
    scaler_scale = z['scale']
    print(f"Loaded Scaler successfully.")
else:
    print("NO SCALER FOUND!")
    scaler_mean = 0.0
    scaler_scale = 1.0

class_df = pd.read_csv(CLASSES_CSV)
index_to_english = {}
index_to_arabic = {}
for _, row in class_df.iterrows():
    idx = int(row['model_class_index'])
    label = str(row['label_name'])
    index_to_english[idx] = label
    index_to_arabic[idx] = label

print(f'{len(index_to_english)} word classes loaded.')


Loading model...
Model loaded: sequential
Loaded Scaler successfully.
33 word classes loaded.


In [ ]:
mp_hands = mp.solutions.hands
mp_holistic = mp.solutions.holistic
mp_drawing = mp.solutions.drawing_utils
mp_drawing_styles = mp.solutions.drawing_styles

if NUM_FEATURES == 258:
    detector = mp_holistic.Holistic(
        static_image_mode=False,
        model_complexity=1,
        enable_segmentation=False,
        refine_face_landmarks=False,
        min_detection_confidence=0.5,
        min_tracking_confidence=0.5
    )
    print("MediaPipe Holistic detector ready (258 features mode)")
else:
    detector = mp_hands.Hands(
        static_image_mode=False,
        max_num_hands=NUM_HANDS,
        min_detection_confidence=0.6,
        min_tracking_confidence=0.6
    )
    print(f'MediaPipe hand detector ready ({NUM_HANDS} hand(s) mode)')

def extract_landmarks(frame):
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    draw_landmarks = []
    
    if NUM_FEATURES == 258:
        results = detector.process(rgb)
        
        # Pose: 132 features
        if results.pose_landmarks:
            pose = np.array([[lm.x, lm.y, lm.z, lm.visibility] for lm in results.pose_landmarks.landmark], dtype=np.float32).flatten()
            draw_landmarks.append(('pose', results.pose_landmarks))
        else:
            pose = np.zeros(132, dtype=np.float32)

        # Left hand: 63 features
        if results.left_hand_landmarks:
            lh = np.array([[lm.x, lm.y, lm.z] for lm in results.left_hand_landmarks.landmark], dtype=np.float32).flatten()
            draw_landmarks.append(('hand', results.left_hand_landmarks))
        else:
            lh = np.zeros(63, dtype=np.float32)

        # Right hand: 63 features
        if results.right_hand_landmarks:
            rh = np.array([[lm.x, lm.y, lm.z] for lm in results.right_hand_landmarks.landmark], dtype=np.float32).flatten()
            draw_landmarks.append(('hand', results.right_hand_landmarks))
        else:
            rh = np.zeros(63, dtype=np.float32)

        combined = np.concatenate([pose, lh, rh])
        return combined, draw_landmarks

    elif NUM_HANDS == 1:
        results = detector.process(rgb)
        if results.multi_hand_landmarks:
            lm = results.multi_hand_landmarks[0]
            vec = np.array([[p.x, p.y, p.z] for p in lm.landmark], dtype=np.float32).flatten()
            return vec, [('hand', lm)]
        return np.zeros(NUM_FEATURES, dtype=np.float32), []

    else:
        results = detector.process(rgb)
        left_vec = np.zeros(LANDMARKS_PER_HAND, dtype=np.float32)
        right_vec = np.zeros(LANDMARKS_PER_HAND, dtype=np.float32)

        if results.multi_hand_landmarks and results.multi_handedness:
            for hand_lm, handedness in zip(results.multi_hand_landmarks, results.multi_handedness):
                draw_landmarks.append(('hand', hand_lm))
                label = handedness.classification[0].label
                vec = np.array([[p.x, p.y, p.z] for p in hand_lm.landmark], dtype=np.float32).flatten()
                if label == 'Left':
                    left_vec = vec
                else:
                    right_vec = vec

        combined = np.concatenate([left_vec, right_vec])
        return combined, draw_landmarks


MediaPipe Holistic detector ready (258 features mode)


: 

In [ ]:
# ==========================================
# GESTURE FEATURE TOGGLES
# ==========================================
# Set this to False if you want to permanently disable the 
# "Crossed Arms" physical erasing gesture in the future.
ENABLE_GESTURE_ERASE = True  

def run_live_test():
    cap = cv2.VideoCapture(CAMERA_INDEX)
    cap.set(cv2.CAP_PROP_FRAME_WIDTH, CAMERA_WIDTH)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, CAMERA_HEIGHT)

    if not cap.isOpened():
        print('Cannot open camera!')
        return

    print(f'Camera opened. Mode: {NUM_FEATURES} F')
    print('Keyboard: (Q) Quit, (R/C) Clear Sentence, (SPACE) Add Space, (D/BACKSPACE/X) Delete Word.')
    print('Gesture : Cross arms "X" over chest to erase last word.')

    frame_buffer = deque(maxlen=SEQUENCE_LENGTH)
    prediction_history = deque(maxlen=STABILITY_WINDOW)
    
    # -----------------------------------------------------------------
    # INJECTING PRE-FILLED WORDS HERE FOR TESTING DELETION
    # -----------------------------------------------------------------
    sentence_words_en = ["first", "second", "test"] 
    
    current_conf = 0.0
    last_prediction_time = 0.0
    last_confirmed_time = 0.0
    last_erase_time = 0.0
    hands_count = 0

    GREEN, RED, WHITE, BLACK, YELLOW, ORANGE = (0, 200, 0), (0, 0, 200), (255, 255, 255), (0, 0, 0), (0, 220, 220), (0, 140, 255)

    while True:
        ret, frame = cap.read()
        if not ret: break

        frame = cv2.flip(frame, 1)
        h, w = frame.shape[:2]
        now = time.time()

        landmarks, draw_items = extract_landmarks(frame)
        hands_count = sum(1 for item in draw_items if item[0]=='hand')
        frame_buffer.append(landmarks)

        # --- ERASE GESTURE ---
        if ENABLE_GESTURE_ERASE and NUM_FEATURES == 258 and (now - last_erase_time) > 2.0:
            ls_x, ls_y = landmarks[11*4], landmarks[11*4+1]
            rs_x, rs_y = landmarks[12*4], landmarks[12*4+1]
            lw_x, lw_y = landmarks[15*4], landmarks[15*4+1]
            rw_x, rw_y = landmarks[16*4], landmarks[16*4+1]
            
            if ls_x != 0 and rs_x != 0 and lw_x != 0 and rw_x != 0:
                d1 = np.sqrt((lw_x - rs_x)**2 + (lw_y - rs_y)**2)
                d2 = np.sqrt((rw_x - ls_x)**2 + (rw_y - ls_y)**2)
                if d1 < 0.2 and d2 < 0.2:
                    if sentence_words_en:
                        sentence_words_en.pop()
                        cv2.putText(frame, "WORD ERASED!", (w//2 - 150, h//2), cv2.FONT_HERSHEY_SIMPLEX, 1.5, RED, 4)
                        print("❌ Gesture Erased last word.")
                    last_erase_time = now
                    frame_buffer.clear()

        # --- Draw landmarks ---
        for item_type, lm in draw_items:
            if item_type == 'hand':
                mp_drawing.draw_landmarks(frame, lm, mp_hands.HAND_CONNECTIONS, mp_drawing_styles.get_default_hand_landmarks_style())
            elif item_type == 'pose':
                mp_drawing.draw_landmarks(frame, lm, mp_holistic.POSE_CONNECTIONS, mp_drawing_styles.get_default_pose_landmarks_style())

        # --- Predict ---
        current_word_en = ''
        if len(frame_buffer) == SEQUENCE_LENGTH and (now - last_prediction_time) >= PREDICTION_INTERVAL:
            last_prediction_time = now

            seq = np.array(list(frame_buffer), dtype=np.float32)
            seq = (seq - scaler_mean) / scaler_scale
            seq = np.expand_dims(seq, axis=0)

            if np.sum(np.any(seq[0] != 0, axis=1)) >= SEQUENCE_LENGTH * 0.3:
                proba = model(seq, training=False).numpy()[0]
                pred_idx = np.argmax(proba)
                pred_conf = proba[pred_idx]
                pred_word_en = index_to_english.get(pred_idx, '?')

                if pred_conf >= CONFIDENCE_THRESHOLD:
                    current_word_en = pred_word_en
                    prediction_history.append(pred_word_en)

                    if len(prediction_history) == STABILITY_WINDOW and len(set(prediction_history)) == 1 and (now - last_confirmed_time) >= COOLDOWN_TIME:
                        sentence_words_en.append(current_word_en)
                        last_confirmed_time = now
                        prediction_history.clear()
                        print(f'✅ Confirmed: "{current_word_en}" ({pred_conf:.1%})')
                else:
                    prediction_history.clear()

        # --- UI ---
        cv2.rectangle(frame, (0, h - 50), (w, h), BLACK, -1)
        sentence_en = ' '.join(sentence_words_en) if sentence_words_en else '(sentence)'
        cv2.putText(frame, f'EN: {sentence_en}', (15, h - 20), cv2.FONT_HERSHEY_SIMPLEX, 0.7, WHITE, 2)
        cv2.imshow('ArSL Live Sandbox Test', frame)

        # --- Keyboard ---
        key = cv2.waitKey(1) & 0xFF
        if key == ord('q'): break
        elif key in [ord('r'), ord('c')]: 
            sentence_words_en.clear()
            prediction_history.clear()
        elif key == 32: sentence_words_en.append(' ')
        elif key in [8, 127, ord('d'), ord('x')]:
            if sentence_words_en: 
                print(f"🗑️ Keyboard Erased: {sentence_words_en.pop()}")

    cap.release()
    cv2.destroyAllWindows()
    print("Test Ended.")

# >>> RUN IT <<<
run_live_test()


Camera opened. Mode: 258 F
Keyboard: (Q) Quit, (R/C) Clear Sentence, (SPACE) Add Space, (D/BACKSPACE/X) Delete Word.
Gesture : Cross arms "X" over chest to erase last word.


Webcam frame per second test


In [ ]:
import cv2
import time
from imutils.video import WebcamVideoStream

print("🎥 Starting pure camera test (No AI)...")

# Turn on the camera
vs = WebcamVideoStream(src=0).start()
time.sleep(2.0) 

prev_time = time.time()
fps = 0 # Start at 0

while True:
    frame = vs.read()
    if frame is None:
        continue

    # 1. Calculate the live Frame Rate (Safely!)
    curr_time = time.time()
    time_diff = curr_time - prev_time
    
    if time_diff > 0:
        fps = 1.0 / time_diff
        
    prev_time = curr_time

    # 2. Draw the FPS on the top-left corner
    cv2.putText(frame, f"FPS: {int(fps)}", (10, 40), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 3)
    cv2.putText(frame, "HARDWARE TEST (NO AI)", (10, 80), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)

    # 3. Show the video
    cv2.imshow('Pure Camera Test', frame)

    # Press 'q' to quit
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# Clean up safely
cv2.destroyAllWindows()
vs.stop()
print("🛑 Camera shut down safely.")


🎥 Starting pure camera test (No AI)...


In [ ]:
import cv2
import time
import numpy as np
import mediapipe as mp
from imutils.video import WebcamVideoStream
from tensorflow.keras.models import load_model

# ==========================================
# 1. SETUP (CHANGE YOUR MODEL NAME HERE)
# ==========================================
MODEL_PATH = "arsl_word_lstm_model_best_v1.h5"  # <-- Make sure this matches your file!
print(f"🧠 Loading brain: {MODEL_PATH}")
model = load_model(MODEL_PATH)

mp_holistic = mp.solutions.holistic
sequence = []
current_prediction = "Waiting for 30 frames..."

# ==========================================
# 2. FEATURE EXTRACTION (Exactly 258 points)
# ==========================================
def extract_258_features(results):
    # Pose (33 points * 4 values = 132)
    if results.pose_landmarks:
        pose = np.array([[res.x, res.y, res.z, res.visibility] for res in results.pose_landmarks.landmark]).flatten()
    else:
        pose = np.zeros(132)
        
    # Left Hand (21 points * 3 values = 63)
    if results.left_hand_landmarks:
        lh = np.array([[res.x, res.y, res.z] for res in results.left_hand_landmarks.landmark]).flatten()
    else:
        lh = np.zeros(63)
        
    # Right Hand (21 points * 3 values = 63)
    if results.right_hand_landmarks:
        rh = np.array([[res.x, res.y, res.z] for res in results.right_hand_landmarks.landmark]).flatten()
    else:
        rh = np.zeros(63)
        
    # 132 + 63 + 63 = 258 exactly!
    return np.concatenate([pose, lh, rh])

# ==========================================
# 3. LIVE THREADED CAMERA LOOP
# ==========================================
print("🎥 Starting threaded camera...")
vs = WebcamVideoStream(src=0).start()
time.sleep(2.0) # Let camera warm up
prev_time = time.time()

with mp_holistic.Holistic(min_detection_confidence=0.5, min_tracking_confidence=0.5) as holistic:
    while True:
        frame = vs.read()
        if frame is None:
            continue

        # Calculate live FPS
        curr_time = time.time()
        fps = 1 / (curr_time - prev_time)
        prev_time = curr_time

        # Convert colors for MediaPipe
        image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        image.flags.writeable = False
        results = holistic.process(image)

        # Extract features and build the 30-frame sequence
        keypoints = extract_258_features(results)
        sequence.append(keypoints)
        sequence = sequence[-30:] # Only keep the most recent 30 frames

        # Predict when we have a full sequence
        if len(sequence) == 30:
            res = model.predict(np.expand_dims(sequence, axis=0), verbose=0)[0]
            class_id = np.argmax(res)
            confidence = res[class_id]
            
            # Only update if the AI is at least 70% sure
            if confidence > 0.70:
                current_prediction = f"Class {class_id} ({confidence*100:.0f}%)"

        # Draw the FPS and Prediction on the screen
        cv2.putText(frame, f"FPS: {int(fps)}", (10, 40), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
        cv2.putText(frame, f"AI Says: {current_prediction}", (10, 80), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2)

        # Show the video feed
        cv2.imshow('Barebones Live Test', frame)

        # Press 'q' to quit
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

# ==========================================
# 4. SHUTDOWN
# ==========================================
cv2.destroyAllWindows()
vs.stop()
print("🛑 Camera shut down safely.")
